# 2-clean&filter

In [ ]:
# TODO: vérifier si tout OK pour fusion id_acteur/id_orateur/nom_orateur. notamment si ils correspondent bien au meme point
# TODO: vérif comparaison de id acteur =! id orateur

In [1]:
# TODO: vérifier les modifs RN :

# ======================
# TODO: AFFILIATIONS :
# ======================
# TODO: renvoi de la derniere affiliation connue de df_deputes ?
# TODO: possible de checker contre groupeAbrev
# TODO : explorer les affiliation manquantes pour identifier les cas limites
# Notamment regarder ceux qui n'ont pas d'affiliation mais bien un groupeAbrev
# Et aviser si on veut forcer l'affiliation à la derniere connue dans df_deputes
# En réalité sans doute des membres du gouvernement, donc toujours le même souci
# de décision à prendre selon l'usage qu'on veut faire des données
# -> ok avec Vincent, ça se défend sans pb.
# -> mais soucis possible si reste encore des non affiliés ou pers ext.
# -> on a moyen de choper le fait qu'ils sont membres du gouv autrement, non ?
# -> genre une autre variable sur leur statut, etc.


# Avoir plusieurs variables
# une d'info gouv vs députés

# pour affiliation
# une des députés (le reste en missing) = ce que l'on a > BALEC
# une des députes + membres gouv = les afficher en tant que tel comme "groupe" > ELLE QU'ON VEUT
# une des députés + ancienne affiliation des membres gouv = compléter les missing > BALEC

# ==============================
# vérif et possibles soucis :
# ==============================

# TODO: matthias : check les cas particuliers.
# TODO : léo, voir ces machins avec matthias ensuite pour clarifier.
# Et voir pourquoi passé par un isin plutôt que ==

# Créer une nouvelle variable d’affiliation politique par groupe parlementaire + gouvernement séparé
# df["groupe&gvt_affiliation"] = df["groupe_députés_affiliation"].fillna("GVT")
# TODO: LM vérifier ça avec matthias : on est sur que les NA = gouv ?
# genre y a pas plein d'autres cas interv extérieurs etc ?
# -> puis genre tous les cas de membre du gouv identifiés avec ancienne affiliation si on fait ?
# possible autre moyen de choper :
# -> oui peut-être : ID mandat en -1 semble souvent = ministre (pas rapporteur, etc.)
# -> qualite_orateur et semble également un bon indicateur : ministre, rapporteur, etc. (mais pas que)
# -> code parole ne colle pas (avis gvt, avis com etc, mais couvre pas leurs autres interventions)

# TODO : CONCLU RDV AVEC MATTHIAS : FAIRE CATÉGORIE DEPUTÉS + GVT, ÇA SERA NOTRE VARIABLE AFFILIATION PRINCIPALE

# TODO : pour gestion possible des cas limites (traitement post affiliation RN vs gauche)
# possible de renvoyer à la toute fin une liste des gens qui ont plusieurs affiliation dans le temps
# (y compris sans étiquette) pour pouvoir gérer au cas par cas le choix de renvoi de l'affiliation.

# "bonne" == "groupe&gvt_affiliation"

# TODO: POST RDV => ON A BIEN UN PB, TOUTES LES MISSINGS VALUES DEVIENNENT GVT (PA0, "UN DÉPUTÉ MACHIN, ETC.")
# DONC BIEN L'ENJEU D'IDENTIFIER LA FONCTION/STATUT, OU DE GÉRER AU CAS PAR CAS, PAR EXEMPLE IF ID != PA0 (ET =! PA-, ETC.)
# PA- -> aussi des sénateurs ? etc.
# Attention : qualité suffit pas seul (sans lire) == y a aussi des gens qui sont directeur de radio nova, etc.
# qualité -> if ministre ou secrétaire d'État in qualite_orateur -> GVT
# + vérif à la mano.



## 2.1 pré-nettoyage, pré-filtrage et pré-recodages

In [113]:
import pandas as pd
import re

# Charger le df concaténé des deux législatures
df = pd.read_csv(
    "../data/interim/extract_15_16_concat.csv",
    low_memory=False,
    dtype={
        "id_orateur": str  # éviter identification en float avant d'avoir ajouté le "PA"
    },
)

print("Shape du df chargé : ", df.shape)

# ==============================
# Pré-nettoyage et pré-filtrage
# ==============================
"""
nb : précision choix 
- exclusion président.e :
role_debat n'est pas bien identifié, utiliser nom_orateur
(avant de le recoder/nettoyer car sinon risque perte par remplacement)
- id_acteur vs id_orateur :
certains cas (~3000) id_orateur plus précis (un code PA) que id_acteur qui a PA0
Mais en fait ce sont des 100% interruptions avec quasi toujours plusieurs locuteurs.
id_orateur en renvoie (mal) un seul -> on préfère garder le PA0 (neutre)
TODO : Rares exceptions avec Dupont-morretti seul, etc. -> mais bon…
"""

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
df = df[~df["nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# Ne garder que le code style NORMAL
df = df[df["code_style"] == "NORMAL"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["code_parole"] = df["code_parole"].fillna("non_précisé")

# Garder une trace de la longueur des interventions brutes
df["len_texte_brut"] = df["texte"].str.len()

# Stabiliser le id_orateur pour être au format AN
df["id_orateur"] = "PA" + df["id_orateur"]
# Remplacer les valeurs manquantes de id_acteur par id_orateur quand disponible
df["id_acteur"] = df["id_acteur"].combine_first(df["id_orateur"])

# ===========================================================
# Récupérer et nettoyer les noms les plus fréquents
# pour chaque id_acteur sauf PA0 et les id_acteur manquants
# ===========================================================

# ========== Recoder par noms les plus fréquents ==========

# Nom le plus fréquent
most_frequent_name = df.groupby("id_acteur")["nom_orateur"].agg(
    lambda x: x.dropna().mode().iloc[0] if x.dropna().size > 0 else None
)  # version plus stable que value_counts().idxmax() en cas d'ex-aequo


# Renvoyer le nom le plus fréquent sauf si id_acteur == PA0 ou id_acteur est manquant
# Limite de la fonction : invisibilise les rares cas d'interventions
# mal identifiées par leur PA, mais qui ont le bon nom
# (ici le nom majoritaire sera renvoyé)
def get_most_frequent_name(row):
    """
    Récupération de la forme la plus fréquente du nom,
    uniquement pour acteurs différents de PA0.
    if PA0 : nom brut, else : nom le plus fréquent pour cet id.
    /!\ Limite de la fonction : invisibilise les rares cas d'interventions
    mal identifiées par leur PA, mais qui ont le bon nom (ici le nom majoritaire sera renvoyé)
    """
    if row["id_acteur"] == "PA0" or pd.isna(row["id_acteur"]):
        return row["nom_orateur"]
    return most_frequent_name.get(row["id_acteur"], row["nom_orateur"])


df["nom_orateur_clean"] = df.apply(get_most_frequent_name, axis=1)

# ========== Nettoyer les noms d'orateurs ==========


def nettoyer_nom(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes
    texte = texte.replace("’", "'")
    return texte


df["nom_orateur_clean"] = df["nom_orateur_clean"].apply(nettoyer_nom)

print("Shape du df après pré-nettoyage et pré-filtrage : ", df.shape)


Shape du df chargé :  (1128128, 29)
Shape du df après pré-nettoyage et pré-filtrage :  (683680, 31)


## 2.2 Match des infos sur les députés (données datan)

### 2.2.1 Match des infos générales

In [114]:
# ==============================
# MATCH DONNÉES DÉPUTÉS
# ==============================

df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")

# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(
    columns=[
        "mail",
        "twitter",
        "facebook",
        "website",
        "active",
        "scoreParticipationSpecialite",
        "datePriseFonction",
        "groupe",
        "naissance",
    ]
)

# ======Fusion des données députés======

print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merger et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="id_acteur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion données députés:", df.shape)

shape avant fusion: (683680, 31)
shape après fusion données députés: (683680, 48)


### 2.2.2 Match temporel des affiliations

In [158]:
# ======================================================
# RECODAGE ET MATCH TEMPOREL DES AFFILIATIONS PARTISANES
# cf. affiliation lors de telle prise de parole
# ======================================================


# ========== Recodage des dénominations de groupes ==========
"""
nb : ici choix de recoder avec les nom des partis,
car ils sont moins sensible aux évolutions marginales de noms,
même si en réalité les groupes parlementaires sont + larges que les partis
et peuvent servir à accueillir des NI d'étiquettes diverses
"""

# Lecture du fichier d'affiliation par périodes
df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format dégueu

# Recodage des partis pour stabilité temporelle des noms
recodage = {
    "RE": "REN",
    "EPR": "REN",
    "LAREM": "REN",
    "MODEM": "DEM",
    "SOC": "SOC-A",
    "NG": "SOC-A",
    "LFI-NUPES": "LFI",
    "FI": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "GDR",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
    # "DEM": "DEM",
}

# Application du recodage des noms de partis au df d'affiliation
df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(recodage)

# ========== Match temporel des affiliations ==========

"""
nb : Plutôt qu'un merge foireux, parti sur un lookup ligne‑à‑ligne
(= pb des orateurs non députés qui étaient pas présents, etc.)
Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb
nb : attention aux bornes temporelles (cf.normalize() pour ignorer l'heure)
"""

# préparation des dates
df["dateSeance_ts"] = pd.to_datetime(
    df["dateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# aviser si jamais besoin un jour de traiter des affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


# Fonction de recodage temporel des affiliations
def get_parti_for_row(row):
    """
    Retourne l'affiliation partisane recodée correspondant à la date de séance.

    La fonction :
    - lit `id_acteur` (assimilé à `mpId`) et `dateSeance_ts` sur la ligne ;
    - parcourt les périodes d'affiliation de ce député (si présent dans aff_by_mp);
    - renvoie `parti_recod` si `dateSeance_ts` (normalisée au jour) est comprise
    entre `dateDebut` et `dateFin` (bornes incluses).
    """
    mp = row.get("id_acteur")  # correspond au mpId
    # gérer le cas des orateurs non députés ou autre type intervention
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("dateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        # attention : .normalize() pour ignorer l'heure car sinon hors des bornes de fin
        if rec["dateDebut"] <= ts.normalize() <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# application du match temporel
df["affiliation_mandat_députés"] = df.apply(get_parti_for_row, axis=1)

# Pas parfait mais pour avoir une idée :
print(
    "affectés :",
    df["affiliation_mandat_députés"].notna().sum(),
    # Eux on sait pas (pas députés, autre code parole intervention, etc.)
    "| non affectés :",
    df["affiliation_mandat_députés"].isna().sum(),
)


affectés : 563127 | non affectés : 120553


### 2.2.3 Fallback des affiliations manquantes

In [ ]:
# ============================================================
# GESTION AFFILIATIONS MANQUANTES et membres du gouvernement
# - Fallback pour les affiliations manquantes
# - Gestion des cas limites (RN, etc.)
# - Création catégorie "GVT" pour les membres du gouvernement

# TODO: vérifier et acter du fallback
# possible fallback : affiliation la plus récente ?
# ============================================================

# TODO : Comprendre pq Bruneel = ["PA720546"] est laissé en Valeur manquante sur une intervention le 9 janvier 2023 et Boyer = ["PA720546"] sur intervention du 7 novembre 2020
# TODO : possible risque nan (ou c'est bon c'est NI ou autre) lors de la première journée mandature ?

In [159]:
# ========= Gestion cas limites RN ==========

# Recodage des RN de la XVe législature au bloc RN
# nb = choix = initialement en NI car pas assez nombreux pour former un groupe

liste_NI_RN = [
    "PA720822",  # Bruno Bilde
    "PA720668",  # Sébastien Chenu
    "PA720468",  # Emmanuel Blairy
    "PA720614",  # Marine Le Pen
    "PA719436",  # Nicolas Meizonnet
    "PA720802",  # Catherine Pujol
    "PA719608",  # Emmanuelle Ménard, rattachée au RN entre 2017 et 2022 mais plus entre 2022 et 2024
    "PA720606",  # Ludovic Pajot
    "PA606212",  # Gilbert Collard
    "PA720798",  # Louis Aliot
]

# Date seuil : fin de la 15e législature
date_seuil = pd.Timestamp("2022-06-21")

# Condition combinée :
condition_NI_RN = (df["id_acteur"].isin(liste_NI_RN)) & (
    df["dateSeance_ts"].dt.normalize() < date_seuil
)  # dt.normalize() pour ignorer l'heure et éviter soucis de bornes


# Application de la modalité uniquement pour les lignes correspondant à la condition
df.loc[condition_NI_RN, "affiliation_mandat_députés"] = "RN"

# Vérification
print("Lignes recodées RN :", condition_NI_RN.sum())
print(
    "Affiliation recodées pour",
    df.loc[condition_NI_RN, "id_acteur"].nunique(),
    "id_acteur uniques",
)

# vérification des cas sans affiliation :
print("Ceci ne modifie pas nombre sans affiliation : simple recodage NI vers RN")


Lignes recodées RN : 6701
Affiliation recodées pour 10 id_acteur uniques
Ceci ne modifie pas nombre sans affiliation : simple recodage NI vers RN


In [160]:
# ========= Cas limites GOUV =============

# vérification des cas sans affiliation :
print(
    "Nombre d'id_acteur uniques sans affiliation :",
    df[df["affiliation_mandat_députés"].isna()]["id_acteur"].nunique(),
)
print("\nValeur counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation_mandat_députés"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)

# Vérifier les cas où affiliation n'est pas nulle mais avec qualité orateur spécifique
# = membres du gouv mais qui sont députés et flaguent donc avec une affiliation députés

mask_affil_with_qualite = df["affiliation_mandat_députés"].notna() & df[
    "qualite_orateur"
].str.contains("ministre|garde des sceaux|secrétaire d'État", case=False, na=False)

print(
    "Nombre de lignes avec affiliation ET qualité gouvernementale :",
    mask_affil_with_qualite.sum(),
)
print("\nAffiliations pour ces cas :")
print(
    df.loc[mask_affil_with_qualite, "affiliation_mandat_députés"].value_counts().head()
)

print("\nExemples de ces lignes :")
print(
    df.loc[
        mask_affil_with_qualite,
        ["nom_orateur_clean", "qualite_orateur", "affiliation_mandat_députés"],
    ]
    .drop_duplicates()
    .head()
)


Nombre d'id_acteur uniques sans affiliation : 234

Valeur counts des id_acteur sans affiliation (top):
id_acteur  nom_orateur_clean     
PA607846   M. Gérald Darmanin        8432
PA773443   M. Éric Dupond-Moretti    6799
PA330357   M. Olivier Dussopt        5831
PA331481   M. Bruno Le Maire         4421
PA717161   Mme Élisabeth Borne       4139
Name: count, dtype: int64
Nombre de lignes avec affiliation ET qualité gouvernementale : 1367

Affiliations pour ces cas :
affiliation_mandat_députés
REN      1131
HOR       108
UDI        77
DEM        39
SOC-A      11
Name: count, dtype: int64

Exemples de ces lignes :
        nom_orateur_clean                                    qualite_orateur  \
15985   M. Franck Riester                             ministre de la culture   
15987   M. Franck Riester                                           ministre   
21533  M. Olivier Dussopt  secrétaire d’État auprès du ministre de l’acti...   
38580    M. Gabriel Attal  secrétaire d’État auprès du minist

In [ ]:
# ========== Gestion membres GVT ==========

# TODO: renvoyer les membres du gouv à une catégorie "GVT" ou "GOUV" pour les différencier des députés, même sans affiliation partisane
# Y COMPRIS QUAND À UN MANDAT ET DONC DÉJÀ UNE AFFILIATION"

"""
nb : tracabilité
/!\ ici on veut récup membres du gouv, souvent en sans affiliation
mais on veut aussi forcer leur etiquette gvt même quand ils ont une affiliation de député
(ex : ministre qui est aussi député)

Logique de recodage :
Recoder membres GVT, uniquement si != PA0 (= garder cohérence avec cas précédents)
si une des conditions suivantes est vérifiée,
- ministre -> ok, 96 personnes pour 130 qualité, mais exclure le cas de Justin Trudeau et 19 cas PA0
- garde des sceaux (pas toujours co-qualifié de ministre) : ok, 2 bien Dupond-Moretti / Belloubet (même si 10 PA0)
- secrétaire d’État -> 40 personnes pour 53 qualité correspondantes, OK (2 PA0)
= basé sur la lecture des résultats de :
df[df["affiliation_mandat_députés"].isna()]["qualite_orateur"].value_counts()

-> mais il faut exclure "Premier ministre du Canada" -> 2 occurences 
Autre option : exclure des PA PA-107309 = Justin Trudeau, Premier ministre du Canada
"""

mask_gvt = (
    df["qualite_orateur"].str.contains(
        "ministre|garde des sceaux|secrétaire d[’']État",
        case=False,
        na=False,
        regex=True,
    )
    & (df["id_acteur"] != "PA0")
    & (df["id_acteur"] != "PA-107309")
)  # exclure Justin Trudeau, "Premier ministre du Canada"

# ========== Création nouvelle variable avec GVT ==========
df["affiliation"] = df["affiliation_mandat_députés"]  # conserve l'affiliation initiale
df.loc[mask_gvt, "affiliation"] = "GVT"  # variable d'analyse principale

# Vérification
print("Lignes recodées GVT :", mask_gvt.sum())
print(
    "Affiliation recodées pour",
    df.loc[mask_gvt, "id_acteur"].nunique(),
    "id_acteur uniques",
)

# vérification des cas sans affiliation :
print(
    "Nombre restant d'interventions sans affiliation :",
    df["affiliation"].isna().sum(),
)
print(
    "Nombre restant d'id_acteur uniques sans affiliation :",
    df[df["affiliation"].isna()]["id_acteur"].nunique(),
)
print("\nValue counts des id_acteur sans affiliation (top):")
print(
    df[df["affiliation"].isna()][["id_acteur", "nom_orateur_clean"]]
    .value_counts()
    .head()
)

# TODO: géréer les cas limites gouv quand sont rapporteurs, etc. (darmanin, EDM, etc.)

Lignes recodées GVT : 110358
Affiliation recodées pour 109 id_acteur uniques
Nombre restant d'interventions sans affiliation : 12073
Nombre restant d'id_acteur uniques sans affiliation : 152

Value counts des id_acteur sans affiliation (top):
id_acteur  nom_orateur_clean                
PA0        Un député du groupe LR               1125
           Plusieurs députés du groupe LR       1036
           Un député du groupe LaREM             512
           Un député du groupe RN                462
           Plusieurs députés du groupe LaREM     430
Name: count, dtype: int64


# ici léo pour demain !!!!

In [170]:
# TODO: reprendre ici !

In [149]:
missing_affil = df[df["affiliation"].isna()][
    ["id_acteur", "nom_orateur_clean", "qualite_orateur"]
]
missing_affil = missing_affil[missing_affil["id_acteur"] != "PA0"]
missing_affil

,id_acteur,nom_orateur_clean,qualite_orateur
378,PA-121339,Mme Olivia Grégoire,présidente de la commission spéciale
2577,PA-121359,M. Christophe Carval,NaN
2578,PA-121379,M. François Dos Santos,NaN
2579,PA-121369,M. Christophe Ramaux,NaN
2581,PA-121369,M. Christophe Ramaux,NaN
...,...,...,...
683657,PA-125239,M. Didier Quercioli,NaN
683658,PA-125199,Mme Catherine Perret,NaN
683660,PA-125239,M. Didier Quercioli,NaN
683661,PA-125209,M. Martial Crance,NaN


In [150]:
missing_affil["nom_orateur_clean"].value_counts()

nom_orateur_clean
M. Jean-Paul Delevoye      79
M. Pierre Moscovici        59
Mme Olivia Grégoire        41
M. Lyes Louffok            26
M. Erwan Lecœur            21
                           ..
Mme Dominique Vérien        1
Mme Laurence Rossignol      1
M. François-Noël Buffet     1
M. Gabriel Attal            1
Mme Aurore Bergé            1
Name: count, Length: 139, dtype: int64

In [ ]:
missing_affil["nom_orateur_clean"].value_counts()

## Compléter les affiliations manquantes quand un ancien député ou groupe connu

EN COURS !

In [ ]:
## Léo = aviser pour forcer une affiliation
# TODO : aviser si groupe abrev ou si dernière affiliation tout cour ou si dernière affiliation connue (= enc ours) à la date interv -> relou, mais voilà


In [ ]:
# TODO : léo, voir ces machins avec matthias ensuite pour clarifier.
# Et voir pourquoi passé par un isin plutôt que ==

# TODO : aviser pour une variable affiliation forcée

In [ ]:
# # solution temporaire sur 2 cas étranges
# Boyer = ["PA330684"]  # cas similaire sur intervention du 7 novembre 2020

# df.loc[df["id_acteur"].isin(Boyer), "groupe&gvt_affiliation"] = "LR"

# Bruneel = [
#     "PA720546"
# ]  # ici cas étrange sur une intervention le 9 janvier 2023, il a été laissé en valeur manquante alors que GDR

# df.loc[df["id_acteur"].isin(Bruneel), "groupe&gvt_affiliation"] = "GDR"

In [ ]:
# # nb : le groupe "groupeAbrev" vient du fichier complémentaire info députés
# df["groupe_all_affiliation"] = df["affiliation_mandat_députés"].combine_first(
#     df["groupeAbrev"]
# )
# df["groupe_all_affiliation"] = df["groupe_all_affiliation"].replace(recodage)

In [ ]:
# # Reste des cas particuliers à replacer dans leur affiliation au moment de leurs fonctions gouvernementales respectives
# Bachelot = ["PA332"]

# df.loc[df["id_acteur"].isin(Bachelot), "groupe_all_affiliation"] = (
#     "NI"  # NI ou mettre valeur manquante ? pareil pour Philippe, Le Drian, Rousseau
# )

# Vautrin = ["PA267797"]

# df.loc[df["id_acteur"].isin(Vautrin), "groupe_all_affiliation"] = "REN"

# Philippe = ["PA345619"]

# df.loc[df["id_acteur"].isin(Philippe), "groupe_all_affiliation"] = "NI"

# Ledrian = ["PA1872"]

# df.loc[df["id_acteur"].isin(Ledrian), "groupe_all_affiliation"] = "NI"

# Rousseau = ["PA826635"]

# df.loc[df["id_acteur"].isin(Rousseau), "groupe_all_affiliation"] = "NI"

In [ ]:
# df_gvt = df[df["groupe&gvt_affiliation"] == "GVT"]
# df_gvt["groupe_all_affiliation"].value_counts()

groupe_all_affiliation
REN    51514
NI      8432
DEM     4502
HOR      500
Name: count, dtype: int64

## Export

In [15]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning_full.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (cf : adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# PROVISOIRE !! Regrouper les interventions interrompues

In [ ]:
# TODO: À affiner et vérifier la fusion interventions interrompues

# AVISER : pas le cas ici, mais envisager possible gestion des cas NaN
df_interruption = df[df["code_grammaire"].str.contains("INTERRUPTION")]
df_intervention = df[~df["code_grammaire"].str.contains("INTERRUPTION")]
# Si il fallait s'en assurer :
# is_interruption = df["code_grammaire"].str.contains("INTERRUPTION", na=False)
# df_interruption = df[is_interruption]
# df_intervention = df[~is_interruption]

# assert len(df_interruption) + len(df_intervention) == len(df), (
#     f"Lignes perdues lors du split ! "
#     f"{len(df)} ≠ {len(df_interruption)} + {len(df_intervention)} "
#     f"(NaN dans code_grammaire : {df['code_grammaire'].isna().sum()})"
# )

# ordinal_prise semble plus précis au niveau des intervenants
# = est constant quand interrompu là où les ordres obsolu et ptsodj changent
group_keys = ["uid", "dateSeance_ts", "id_acteur", "ordinal_prise"]

# agréger : concat texte, sommer longueur, garder premières infos utiles
agg = {
    "texte": lambda s: " ".join(s.dropna().astype(str)).strip(),
    "len_texte_brut": "sum",
    "code_parole": lambda s: ", ".join(sorted(set(s.dropna().astype(str)))),
    "id_syceron": lambda s: s.dropna().unique().tolist(),
    # "ordre_absolu_seance": "first", # list pour garder l'ordre des prises ?
    # "nom_orateur": "first",
    # "qualite_orateur": "first",
    # "id_orateur": "first",
    # "stime": "first",
}

# ajouter 'first' pour toutes les autres colonnes non clés/non déjà agrégées
for c in df_intervention.columns:
    if c not in group_keys and c not in agg:
        agg[c] = "first"

# Regroupe les interventions par clés communes et agrège les colonnes définies dans `agg`
df_intervention_grouped = (
    df_intervention.groupby(group_keys, dropna=False).agg(agg).reset_index()
)

# Recolle les interventions regroupées avec les interruptions
# puis aligne les colonnes sur le format d'origine
df_concat = pd.concat([df_intervention_grouped, df_interruption], ignore_index=True)[
    df_interruption.columns
]

# Retrier dans l'ordre chronologique et d'affichage de la séance
# nb : ici ok car gardé seulement first pour ordre_absolu_seance
# mais modif si jamais on avait gardé la liste complète des ordres
df_concat = df_concat.sort_values(
    by=["dateSeance_ts", "valeur_ptsodj", "ordre_absolu_seance"]
).reset_index(drop=True)

print(
    f"Regroupement des interventions interrompues \n"
    f"avant: {len(df)} | après: {len(df_concat)} "
    f"(interventions: de {len(df_intervention)} → à {len(df_intervention_grouped)}, "
    f"interruptions: {len(df_interruption)})"
)

In [18]:
# TODO : len_texte_brut sera à réajouter pour interv groupée
# , car là on à la trace de la longueur des interventions avant regroupement
# donc voir pour une fois regroupé ?
# Surtout, renvoyer l'info de longueur dans df regroupé peut porter à confusion ?
# ATTEND : J'EN FAIS DÉJÀ UNE SOMME NON :     "len_texte_brut": "sum",
# aviser, quitte a préciser que c'est une nouvelle col.

In [20]:
# Export du csv concat nettoyé
df_concat.to_csv("../data/interim/data_cleaning_grouped.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

### EXPLORATION

In [21]:
# TODO : aller voir parce que ça regroupe quand meme des trucs qui
# ont pas le même code parole
# donc voir le pourquoi du comment
# MAIS ON S'EN COGNE UN PEU SUR LE PRINCIPE ?
# ENFIN AVISER QUE JUSTE LES AVIS GOUV SOIT PAS REGROUPÉS
# AVEC UNE PRISE PAROLE PLUS LARGE ?
# CHANGE RIEN DE DRAMATIQUE SANS DOUTE.

df_concat["code_parole"].value_counts()[:-10]

code_parole
non_précisé                                 287786
PAROLE_1_2                                   92122
PAROLE_1_2, non_précisé                      15040
AVIS_COM_1_20                                10824
AVIS_GVT_1_20                                10097
AVIS_GVT_1_20, PAROLE_1_2                     3371
AVIS_COM_1_20, non_précisé                    2265
AVIS_COM_1_20, PAROLE_1_2                     1787
AVIS_GVT_1_20, non_précisé                    1096
AVIS_COM_1_20, PAROLE_1_2, non_précisé         684
AVIS_GVT_1_20, PAROLE_1_2, non_précisé         453
AVIS_COM_1_20, AVIS_GVT_1_20                    14
AVIS_COM_1_20, AVIS_GVT_1_20, PAROLE_1_2         5
Name: count, dtype: int64

In [22]:
# Vérifier si code_parole varie au sein d'un même groupe
check = df_intervention.groupby(group_keys, dropna=False)["code_parole"].nunique()
print("Groupes avec code_parole non constant :", (check > 1).sum())

Groupes avec code_parole non constant : 24724


In [23]:
test = df["qualite_orateur"].value_counts()

In [24]:
test.head(150)

qualite_orateur
ministre                                                                     47843
rapporteur                                                                   30585
rapporteure                                                                  18597
secrétaire d’État                                                            15423
rapporteur général                                                           13692
                                                                             ...  
président de la commission mixte paritaire                                      47
secrétaire d’État chargée de la jeunesse et du service national universel       46
secrétaire d’État chargée de la ville et de la citoyenneté                      46
ministre de l’économie, des finances et de la relance                           46
ministre de la transition écologique et solidaire                               45
Name: count, Length: 150, dtype: int64

# EXPLO

In [25]:
# SI VÉRIF LE FAIRE AVANT NETTOYAGE DES NOMS.

mask_diff_nom = df["nom_orateur"].fillna("__NA__") != df["nom_orateur_clean"].fillna(
    "__NA__"
)

print("Nombre de lignes différentes :", mask_diff_nom.sum())
print("Part des lignes différentes :", round(mask_diff_nom.mean() * 100, 2), "%")

df.loc[mask_diff_nom, ["nom_orateur", "nom_orateur_clean"]].drop_duplicates().head(20)

Nombre de lignes différentes : 15921
Part des lignes différentes : 2.33 %


,nom_orateur,nom_orateur_clean
23,Mme Elsa Faucillon et M. Richard Ramos,Mme Elsa Faucillon et M. Richard Ramos
343,Mme Olivia Gregoire,Mme Olivia Grégoire
471,M. Raphaël Schellenberger et M. Adrien Quatennens,M. Raphaël Schellenberger et M. Adrien Quatennens
672,M. Stéphane Peu et M. François Pupponi,M. Stéphane Peu et M. François Pupponi
706,"M. Nicolas Forissier et M. Roland Lescure, rap...","M. Nicolas Forissier et M. Roland Lescure, rap..."
784,M. M’jid El Guerrab,M. M'jid El Guerrab
785,M. Thibault Bazin et M. Bertrand Pancher,M. Thibault Bazin et M. Bertrand Pancher
970,M. Jean-Pierre Cubertafon et M. Jimmy Pahun,M. Jean-Pierre Cubertafon et M. Jimmy Pahun
1035,Mme Sylvia Pinel et Mme Bénédicte Taurine,Mme Sylvia Pinel et Mme Bénédicte Taurine
1211,M. Edouard Philippe,M. Édouard Philippe
